# Supplementary Figs. 26-30 - generating the simulated mCA samples

The simulated samples used to benchmark both mCA callers are made by **spiking mCAs of known type, size
and cell fraction into real control samples**. This notebook is the generator.

| step | what it does |
|---|---|
| control selection | ranks 36 controls by mean depth and PON-normalised depth CV, splits them 9 training / 9 test, and checks panel coverage of the target arm. **This step is Supplementary Fig. 30** |
| chromosome assignment | `get_clean_chromosomes` picks chromosomes on which the chosen control has no pre-existing event |
| spike-in | `run_simulation` / `calculate_new_vaf_and_depth` shift depth and BAF by closed-form expressions in the cell fraction *f*, then resample with Poisson (depth) and binomial (variant reads) so the output carries realistic noise |

For a heterozygous SNP assigned to the affected haplotype, with cell fraction *f*:

| event | expected total depth | expected BAF, affected vs other haplotype |
|---|---|---|
| GAIN | `depth x (1 + f/2)` | `(1+f)/(2+f)` vs `1/(2+f)` |
| LOSS | `depth x (1 - f/2)` | `(1-f)/(2-f)` vs `1/(2-f)` |
| CN-LOH | `depth` (copy-neutral) | `(1+f)/2` vs `(1-f)/2` |

The grid is 3 event types x 14 cell fractions (0.1 % to 100 %) x 6 length categories (2, 3, 5, 10, 20 Mb,
WholeArm) x 3 geometries (Interstitial, Telomeric, Whole_Arm) across 9 controls - **12,306 simulated
samples in each of the training and test sets**.

The haplotype assignment is recorded per SNP as `Phased_Haplotype`, which is the ground truth the phased
caller is scored against in `Supplementary_Fig_28-29-mCA_phased_caller_benchmarking.ipynb`. The unphased
caller's performance on the same samples is Supplementary Fig. 26.

## Data availability

**This notebook is code-only: its inputs and outputs are data-protected.**

The inputs are the per-SNP BAF files of nine real control samples, deposited under controlled access in
the European Genome-phenome Archive and released to approved researchers via a Data Access Committee,
together with the CNV panel-of-normals matrix.

The outputs inherit those controls' germline genotypes and so cannot be distributed either: a simulated
sample keeps 100 % of its control's heterozygous sites and ~80 % of its BAF values unchanged, because only
the spiked region is modified.

**Every simulation is nonetheless reproducible.** Each seed derives from a `MASTER_SEED` and is recorded
in `simulation_manifest.csv`, so an approved user with access to the nine controls can regenerate any
individual sample exactly.

| file | used for | location |
|---|---|---|
| control `*_SNPs.txt` / per-SNP BAF files | the samples mCAs are spiked into | EGA, controlled access |
| `PON_master_matrix.tsv` | per-sample depth CV, for control selection | not distributed |
| `TWIST_CNV_panel_TE-95031423_h19.bed` | panel coverage of the target arm | Data_files |
| `simulation_manifest.csv` (written) | ground truth and the seed for every simulated sample | not distributed |

The small summary tables the downstream figures actually plot are included, in
`Data_files/mCA_calling/Simulated_data/mCA_simulation_test_set/`.


# Import packages

In [ ]:
# imported packages
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns
import matplotlib.gridspec as gridspec

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'Helvetica'

In [ ]:
# Lists of colors for plots
c0 = (0.76, 0.76, 0.76)
c1 = (1.00, 0.18, 0.33);
c2 = (1.00, 0.23, 0.19);
c3 = (1.00, 0.58, 0.00);
c4 = (1.00, 0.80, 0.00);
c5 = (0.30, 0.85, 0.39);
c6 = (0.35, 0.78, 0.98);
c7 = (0.20, 0.67, 0.86);
c8 = (0.00, 0.48, 1.00);
c9 = (0.35, 0.34, 0.84);
c10 = (0.00, 0.31, 0.57);
c11 = (0.12, 0.29, 0.69);
c12 = (0.17, 0.17, 0.42);
c13 = (1.00, 1.00, 1.00);
c14 = (0.77, 0.04, 0.00);

In [ ]:
#define the colors from colorbrewer2
orange1 = '#feedde'
orange2 = '#fdbe85'
orange3 = '#fd8d3c'
orange4 = '#e6550d'
orange5 = '#a63603'
blue1 = '#eff3ff'
blue2 = '#bdd7e7'
blue3 = '#6baed6'
blue4 = '#3182bd'
blue5 = '#08519c'
green1 = '#edf8e9'
green2 = '#bae4b3'
green3 = '#74c476'
green4 = '#31a354'
green5 = '#006d2c'
grey1 = '#f7f7f7'
grey2 = '#cccccc'
grey3 = '#969696'
grey4 = '#636363'
grey5 = '#252525'
purple1 = '#f2f0f7'
purple2 = '#cbc9e2'
purple3 = '#9e9ac8'
purple4 = '#756bb1'
purple5 = '#54278f'
red1 = '#fee5d9'
red2 = '#fcae91'
red3 = '#fb6a4a'
red4 = '#de2d26'
red5 = '#a50f15'
yellow = '#ffffd4'

# Choose control samples to use for generation of simulated mCA spike-ins

In [ ]:
# ==========================================
# GENOME DATA (HG19)
# ==========================================
hg19_info = {
    'chr1': (249250621, 125000000), 'chr2': (243199373, 93300000), 'chr3': (198022430, 91000000),
    'chr4': (191154276, 50400000),  'chr5': (180915260, 48400000), 'chr6': (171115067, 61000000),
    'chr7': (159138663, 59900000),  'chr8': (146364022, 45600000), 'chr9': (141213431, 49000000),
    'chr10': (135534747, 40200000), 'chr11': (135006516, 53700000), 'chr12': (133851895, 35800000),
    'chr13': (115169878, 17900000), 'chr14': (107349540, 17600000), 'chr15': (102531392, 19000000),
    'chr16': (90354753, 36600000),  'chr17': (81195210, 24000000),  'chr18': (78077248, 17200000),
    'chr19': (59128983, 26500000),  'chr20': (63025520, 27500000),  'chr21': (48129895, 13200000),
    'chr22': (51304566, 14700000),  'chrX': (155270560, 60600000)
}

In [ ]:
def get_clean_chromosomes(sample, control_events, n_chroms=3, seed=8, used_chroms=set()):
    rng = random.Random(seed)
    affected = set(control_events.get(sample, []))

    groups = [
        ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7'],
        ['chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15'],
        ['chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX']
    ]
    
    selected = []
    for group in groups:
        # Prefer: not affected AND not used by another sample
        available = [c for c in group if c not in affected and c not in used_chroms]
        if not available:
            # Fallback: not affected but allow reuse
            available = [c for c in group if c not in affected]
        if not available:
            print(f"  ⚠️ No clean chromosomes in group {group} for {sample}")
            continue
        selected.append(rng.choice(available))
    
    return selected

def get_coordinates(chrom, length_mb, geometry_type, seed, excluded_arms, bed_df):
    """Generate coordinates, avoiding arms with insufficient coverage"""
    rng = random.Random(seed)
    size, centro = hg19_info.get(chrom, (0, 0))
    if size == 0:
        return 0, 0

    arms = [
        ('p', 0, centro, f"{chrom}p"),
        ('q', centro, size, f"{chrom}q")
    ]
    
    # Filter out excluded arms
    valid_arms = [(name, s, e, arm_id) for (name, s, e, arm_id) in arms 
                  if arm_id not in excluded_arms]

    if geometry_type == 'Whole_Arm':
        if not valid_arms:
            return 0, 0
        _, start, end, _ = rng.choice(valid_arms)
        # Check probe coverage for whole arm
        if bed_df is not None:
            probes_in_region = bed_df[(bed_df['chromosome'] == chrom) & 
                                      (bed_df['start'] >= start) & 
                                      (bed_df['start'] <= end)]
            if len(probes_in_region) == 0:
                return 0, 0
            arm_length_mb = (end - start) / 1e6
            min_probes = max(5, int(arm_length_mb * 2))
            if len(probes_in_region) < min_probes:
                return 0, 0
            # Snap to first and last probe positions
            start = int(probes_in_region['start'].min())
            end = int(probes_in_region['end'].max())
        
        return start, end

    length_bp = int(length_mb * 1_000_000)
    
    # For interstitial/telomeric, check if arms are large enough
    large_enough = [(s, e, arm_id) for (n, s, e, arm_id) in valid_arms 
                    if (e - s) > length_bp]
    
    if not large_enough:
        return 0, 0
    
    arm_start, arm_end, arm_id = rng.choice(large_enough)

    if geometry_type == 'Telomeric':
        if arm_start == 0:
            start, end = 0, length_bp
        else:
            start, end = arm_end - length_bp, arm_end
    elif geometry_type == 'Interstitial':
        buffer = 500_000
        min_start = arm_start + buffer
        max_start = arm_end - length_bp - buffer
        if max_start <= min_start:
            min_start, max_start = arm_start, arm_end - length_bp
        if max_start < min_start:
            return 0, 0
        pick_start = rng.randint(min_start, max_start)
        start, end = pick_start, pick_start + length_bp
    else:
        return 0, 0

    # Verify probe coverage
    if bed_df is not None and length_mb > 0:
        probes_in_region = bed_df[(bed_df['chromosome'] == chrom) & 
                                  (bed_df['start'] >= start) & 
                                  (bed_df['start'] <= end)]
        min_probes = max(5, int(length_mb * 2))
        if len(probes_in_region) < min_probes:
            return 0, 0

    return start, end

def calculate_new_vaf_and_depth(current_depth, fraction, mca_type, haplotype_mask, rng):
    if mca_type == 'GAIN':
        expected_total = current_depth * (1 + fraction/2)
        expected_vaf = np.where(haplotype_mask == 1, (1 + fraction)/(2 + fraction), 1/(2 + fraction))
    elif mca_type == 'LOSS':
        expected_total = current_depth * (1 - fraction/2)
        expected_vaf = np.where(haplotype_mask == 1, (1 - fraction)/(2 - fraction), 1/(2 - fraction))
    elif mca_type == 'CNLOH':
        expected_total = current_depth 
        expected_vaf = np.where(haplotype_mask == 1, (1 + fraction)/2, (1 - fraction)/2)

    new_total_depth = rng.poisson(expected_total)
    new_total_depth = np.maximum(new_total_depth, 1)
    new_variant_depth = rng.binomial(new_total_depth, expected_vaf)
    
    return new_total_depth, new_variant_depth

def run_simulation(sample_name, chrom, start_bp, end_bp, mca_type, fraction, 
                   df_snps_orig, df_lrr_orig, sim_id, seed, haplotype_assignment=None):
    
    rng = np.random.RandomState(seed)
    df_snps = df_snps_orig.copy()
    df_lrr = df_lrr_orig.copy()
    
    snp_chrom = chrom if chrom.startswith('chr') else f"chr{chrom}"
    lrr_chrom = chrom if chrom.startswith('chr') else f"chr{chrom}"
    
    # ALL SNPs in the region (for depth/LRR changes)
    mask_all_snps = (df_snps['chromosome'] == snp_chrom) & \
                    (df_snps['position'] >= start_bp) & \
                    (df_snps['position'] <= end_bp)
    
    # Only hets (for BAF changes)
    mask_het_snps = mask_all_snps & \
                    (df_snps['VAF'] > 0.2) & (df_snps['VAF'] < 0.8)

    df_snps['Phased_Haplotype'] = np.nan 

    # 1. Apply BAF shift to hets only
    if mask_het_snps.sum() > 0:
        # Use provided haplotype assignment (shared across CFs in a family)
        # or generate one if not provided (backward compatibility)
        if haplotype_assignment is None:
            haplotype_assignment = rng.binomial(1, 0.5, mask_het_snps.sum())
        df_snps.loc[mask_het_snps, 'Phased_Haplotype'] = haplotype_assignment

        current_total = df_snps.loc[mask_het_snps, 'total_depth'].values
        new_total, new_variant = calculate_new_vaf_and_depth(
            current_total, fraction, mca_type, haplotype_assignment, rng
        )
        df_snps.loc[mask_het_snps, 'total_depth'] = new_total
        df_snps.loc[mask_het_snps, 'variant_depth'] = new_variant
        df_snps.loc[mask_het_snps, 'VAF'] = new_variant / new_total

    # 2. Apply depth shift to homozygous SNPs (NEW)
    mask_hom_snps = mask_all_snps & ~mask_het_snps
    if mask_hom_snps.sum() > 0 and mca_type != 'CNLOH':
        factor = 1 + (fraction / 2) if mca_type == 'GAIN' else 1 - (fraction / 2)
        current_total_hom = df_snps.loc[mask_hom_snps, 'total_depth'].values
        new_total_hom = rng.poisson(current_total_hom * factor)
        new_total_hom = np.maximum(new_total_hom, 1)
        
        # For hom ref (VAF~0): variant_depth stays ~0
        # For hom alt (VAF~1): variant_depth scales with total
        current_vaf_hom = df_snps.loc[mask_hom_snps, 'VAF'].values
        new_variant_hom = rng.binomial(new_total_hom, current_vaf_hom)
        
        df_snps.loc[mask_hom_snps, 'total_depth'] = new_total_hom
        df_snps.loc[mask_hom_snps, 'variant_depth'] = new_variant_hom
        df_snps.loc[mask_hom_snps, 'VAF'] = new_variant_hom / new_total_hom

    # 3. LRR bin file (unchanged)
    mask_lrr = (df_lrr.index.get_level_values('chromosome') == lrr_chrom) & \
               (df_lrr.index.get_level_values('start') >= start_bp) & \
               (df_lrr.index.get_level_values('stop') <= end_bp)
               
    if mca_type != 'CNLOH':
        factor = 1 + (fraction / 2) if mca_type == 'GAIN' else 1 - (fraction / 2)
        noise = rng.normal(1.0, 0.02, size=mask_lrr.sum()) 
        df_lrr.loc[mask_lrr, 'sample_depth_normalised_by_PON'] *= (factor * noise)
        df_lrr.loc[mask_lrr, 'log2ratio'] = np.log2(df_lrr.loc[mask_lrr, 'sample_depth_normalised_by_PON'])
    
    # Save Files
    out_snp_name = f"{sample_name}_{sim_id}_SNPs.txt"
    df_snps.to_csv(os.path.join(output_dir, out_snp_name), sep='\t', index=False, na_rep='NA')

    out_lrr_name = f"{sample_name}_{sim_id}_PON_normalised_read_depths_and_LRR.txt"
    with open(os.path.join(output_dir, out_lrr_name), 'w') as f:
        f.write(f"sample name :\t{sample_name}_SIMULATED\n")
        f.write(f"date of analysis :\t{pd.Timestamp.today().strftime('%d/%m/%Y')}\n")
        f.write(f"produced from code:\tWatson_Simulation_v4.1_Batch_Seeded\n")
        f.write(f"simulation details:\t{sim_id} {start_bp}-{end_bp} seed:{seed}\n\n") # Added Seed to Header
        df_lrr.to_csv(f, sep='\t')

In [ ]:
# ==========================================
# CHECK PANEL COVERAGE FROM BED FILE
# ==========================================
PANEL_BED_FILE = "Data_files/TWIST_CNV_panel_TE-95031423_h19.bed"

def check_arm_coverage_from_bed(bed_file):
    """
    Load BED file and count probes per chromosome arm
    Returns: dictionary of arm coverage and list of excluded arms
    """
    # Load BED file
    bed_df = pd.read_csv(bed_file, sep='\t', skiprows=3, header=None, names=['chromosome', 'start', 'end'])
    
    # Ensure chromosome naming matches (add 'chr' if needed)
    if not bed_df['chromosome'].iloc[0].startswith('chr'):
        bed_df['chromosome'] = 'chr' + bed_df['chromosome'].astype(str)
    
    # Count probes per arm
    arm_coverage = {}
    
    for chrom, (size, centro) in hg19_info.items():
        chrom_probes = bed_df[bed_df['chromosome'] == chrom]
        
        # Count probes in p-arm (before centromere)
        p_arm_probes = len(chrom_probes[chrom_probes['start'] < centro])
        
        # Count probes in q-arm (after centromere)
        q_arm_probes = len(chrom_probes[chrom_probes['start'] >= centro])
        
        arm_coverage[f"{chrom}p"] = p_arm_probes
        arm_coverage[f"{chrom}q"] = q_arm_probes
    
    return arm_coverage, bed_df

print(f"📂 Loading panel coverage from {PANEL_BED_FILE}...")
arm_coverage, bed_df = check_arm_coverage_from_bed(PANEL_BED_FILE)

# Print coverage per arm
print("\n📊 Probe coverage by chromosome arm:")
for chrom in ['chr' + str(i) for i in range(1, 23)] + ['chrX']:
    p_count = arm_coverage.get(f"{chrom}p", 0)
    q_count = arm_coverage.get(f"{chrom}q", 0)
    print(f"  {chrom}: p-arm={p_count:4d} probes, q-arm={q_count:4d} probes")

# Define minimum probes required for whole arm simulation
MIN_PROBES_FOR_WHOLE_ARM = 50  # Adjust threshold as needed

# Identify sparse/uncovered arms
excluded_arms = [arm for arm, count in arm_coverage.items() 
                 if count < MIN_PROBES_FOR_WHOLE_ARM]

print(f"\n❌ Excluded arms (< {MIN_PROBES_FOR_WHOLE_ARM} probes): {excluded_arms}")
print(f"✅ Valid arms for simulation: {len(arm_coverage) - len(excluded_arms)}/{len(arm_coverage)}")

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
base_path = "."
libraries = ['SLX_19285', 'SLX_20125', 'SLX_20127']
pon_matrix_path = "Data_files/mCA_calling/PON_master_matrix.tsv"

# EXCLUDE SAMPLES WITH SOMATIC MCA OR ABNORMAL BAF PLOTS
excluded_samples = ['CNTRL_170_s6', 'CNTRL_203_s5', 'CNTRL_185_s4']

# Master Seed for Reproducibility
MASTER_SEED = 8

# CONTROL SAMPLES WITH MCAs (don't generate the simulated mCAs on these affected chromosomes)
control_events = {'CNTRL_160_s8': ['chr7'],
                  'CNTRL_169_s7': ['chr18'],
                  'CNTRL_174_s3': ['chr4', 'chr7', 'chr13', 'chrX'],
                  'CNTRL_177_s4': ['chrX'],
                  'CNTRL_181_s7': ['chr16'],
                  'CNTRL_182_s4': ['chr2', 'chr3'],
                  'CNTRL_183_s6': ['chr1'],
                  'CNTRL_186_s4': ['chr17'],
                  'CNTRL_188_s7': ['chr7', 'chr17', 'chrX'],
                  'CNTRL_193_s2': ['chr9', 'chrX'],
                  'CNTRL_199_s7': ['chr5'],
                  'CNTRL_163_s6': ['chr5', 'chr16', 'chrX'],
                  'CNTRL_164_s6': ['chrX'],
                  'CNTRL_167_s2': ['chr3', 'chr8'],
                  'CNTRL_171_s2': ['chr2'],
                  'CNTRL_172_s7': ['chr7'],
                  'CNTRL_175_s3': ['chr21'],
                  'CNTRL_180_s3': ['chr14'],
                  'CNTRL_187_s2': ['chr10', 'chr17', 'chrX'],
                  'CNTRL_190_s4': ['chrX'],
                  'CNTRL_191_s7': ['chr4', 'chr14'],
                  'CNTRL_194_s8': ['chr6'],
                  'CNTRL_001_s10': ['chr2', 'chr13', 'chrX'],
                  'CNTRL_002_s8': ['chr1', 'chr8'],
                  'CNTRL_003_s9': ['chr1', 'chr10'],
                  'CNTRL_004_s10': ['chr19', 'chrX'],
                  'CNTRL_005_s9': ['chr6', 'chr9', 'chrX'],
                  'CNTRL_162_s5': ['chr19', 'chrX'],
                  'CNTRL_185_s4': ['chr15'],
                  'CNTRL_189_s4': ['chr6', 'chr15'],
                  'CNTRL_195_s3': ['chr3'],
                  'CNTRL_196_s8': ['chr22'],
                  'CNTRL_198_s2': ['chr3', 'chr6']}

# --- SIMULATION PARAMETERS ---
fractions = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00] # 0.1% to 100%
mca_types = ['GAIN', 'LOSS', 'CNLOH']

# Geometries: 3Mb, 5Mb & 20Mb (Interstitial + Telomeric), plus Whole Arm
simulation_geometries = [
    {'len': 2,  'type': 'Interstitial'},
    {'len': 2,  'type': 'Telomeric'},
    {'len': 3,  'type': 'Interstitial'},
    {'len': 3,  'type': 'Telomeric'},
    {'len': 5,  'type': 'Interstitial'},
    {'len': 5,  'type': 'Telomeric'},
    {'len': 10, 'type': 'Interstitial'},
    {'len': 10, 'type': 'Telomeric'},
    {'len': 20, 'type': 'Interstitial'},
    {'len': 20, 'type': 'Telomeric'},
    {'len': 0,  'type': 'Whole_Arm'} 
]

## TRAINING set of samples

In [ ]:
# ==========================================
# 1. LOAD DATA
# ==========================================
print("⏳ Loading PON Master Matrix (for Noise/CV calculations)...")
if not os.path.exists(pon_matrix_path):
    print(f"❌ Error: PON Matrix not found at {pon_matrix_path}")
    raise SystemExit

# Load Matrix (used ONLY for CV)
pon_df = pd.read_csv(pon_matrix_path, sep='\t', index_col=[0, 1, 2, 3])

# Calculate CV (Noise) from the Matrix
# CV = StdDev / Mean (The mean here is ~1.0, but we calculate it to be precise)
sample_cvs = pon_df.std(axis=0) / (pon_df.mean(axis=0) + 1e-6)

print(f"✅ Loaded Matrix. Calculating Raw Depths for {len(sample_cvs)} samples...")

# ==========================================
# 2. RETRIEVE RAW DEPTHS FROM SOURCE FILES
# ==========================================
sample_metadata = {}

for lib in libraries:
    lib_path = os.path.join(base_path, lib)
    if not os.path.exists(lib_path): continue
    
    # Get all sample folders
    samples = [d for d in os.listdir(lib_path) if os.path.isdir(os.path.join(lib_path, d))]
    
    for s in samples:
        # Skip excluded samples
        if s in excluded_samples:
            continue

        # Only process if this sample is in our PON
        if s not in sample_cvs.index: continue
            
        # Find the raw data file
        cnv_folder = os.path.join(lib_path, s, "CNV_read_depths")
        if not os.path.exists(cnv_folder): continue
        
        # Look for the file
        files = [f for f in os.listdir(cnv_folder) if s in f and f.endswith("sample_normalised_read_depths.txt") and "PON" not in f]
        if not files: continue
        
        # Update the file path for THIS sample
        file_path = os.path.join(cnv_folder, files[0])
        
        try:
            df = pd.read_csv(file_path, sep='\t', skiprows=5)
            df.columns = df.columns.str.strip()
                
            if 'mean_region_depth' in df.columns:
                raw_depth = df['mean_region_depth'].mean()
                
                sample_metadata[s] = {
                    'Library': lib,
                    'Raw_Depth': raw_depth,
                    'CV': sample_cvs[s]
                }
        except:
            pass

# Convert to DataFrame
stats_df = pd.DataFrame.from_dict(sample_metadata, orient='index')

# ==========================================
# 3. SELECT SAMPLES (CLOSEST TO MEDIAN)
# ==========================================
selected_samples = []
print("\n🎯 Selecting Representative Samples (Based on Raw Depth & Matrix CV)...")

for lib in libraries:
    lib_df = stats_df[stats_df['Library'] == lib].copy()
    if lib_df.empty: continue
        
    # Calculate Centroids (Medians)
    median_depth = lib_df['Raw_Depth'].median()
    median_cv = lib_df['CV'].median()
    
    # Normalize for distance calculation
    d_norm = (lib_df['Raw_Depth'] - lib_df['Raw_Depth'].mean()) / lib_df['Raw_Depth'].std()
    c_norm = (lib_df['CV'] - lib_df['CV'].mean()) / lib_df['CV'].std()
    md_norm = (median_depth - lib_df['Raw_Depth'].mean()) / lib_df['Raw_Depth'].std()
    mc_norm = (median_cv - lib_df['CV'].mean()) / lib_df['CV'].std()
    
    # Distance
    lib_df['dist'] = np.sqrt((d_norm - md_norm)**2 + (c_norm - mc_norm)**2)
    
    # Pick top 3
    top_3 = lib_df.sort_values('dist').head(3).index.tolist()
    selected_samples.extend(top_3)
    
    print(f"   📂 {lib}: Median Depth={median_depth:.0f}, Median CV={median_cv:.4f}")
    print(f"      -> Selected: {top_3}")

# ==========================================
# 4. PLOTTING
# ==========================================
fig = plt.figure(figsize=(12, 12))
gs = gridspec.GridSpec(2, 2, height_ratios=[3, 1], hspace=0.3)

# --- MAIN SCATTER ---
ax_main = plt.subplot(gs[0, :])
sns.scatterplot(
    data=stats_df, x='Raw_Depth', y='CV', hue='Library', style='Library',
    s=100, alpha=0.6, edgecolor='k', ax=ax_main
)

# Circles
selected_data = stats_df.loc[selected_samples]
ax_main.scatter(
    selected_data['Raw_Depth'], selected_data['CV'], 
    s=400, facecolors='none', edgecolors='black', linewidth=2.5, zorder=10
)

# Labels
for s in selected_samples:
    row = stats_df.loc[s]
    ax_main.text(
        row['Raw_Depth'], row['CV'], f"  {s}", 
        fontsize=10, fontweight='bold', ha='left', va='center'
    )

ax_main.set_title("Corrected Selection: Raw Depth vs Noise", fontsize=16, fontweight='bold')
ax_main.set_xlabel("Raw Mean Region Depth (Sensitivity)", fontsize=12)
ax_main.set_ylabel("CV (Noise from PON)", fontsize=12)
ax_main.grid(True, linestyle='--', alpha=0.4)

# --- HISTOGRAMS ---
ax_h1 = plt.subplot(gs[1, 0])
sns.kdeplot(data=stats_df, x='Raw_Depth', hue='Library', fill=True, ax=ax_h1, alpha=0.3)
for s in selected_samples:
    ax_h1.axvline(stats_df.loc[s, 'Raw_Depth'], color='k', linestyle='--', alpha=0.5)
ax_h1.set_title("Distribution of Raw Depth")

ax_h2 = plt.subplot(gs[1, 1])
sns.kdeplot(data=stats_df, x='CV', hue='Library', fill=True, ax=ax_h2, alpha=0.3)
for s in selected_samples:
    ax_h2.axvline(stats_df.loc[s, 'CV'], color='k', linestyle='--', alpha=0.5)
ax_h2.set_title("Distribution of Noise (CV)")

plt.savefig("Corrected_Control_Selection.pdf", bbox_inches='tight')
plt.show()

# Export for next step
with open("selected_controls.txt", "w") as f:
    f.write("\n".join(selected_samples))

In [ ]:
# --- CONTROL SAMPLES TO USE ---
# Format: 'SampleName': 'LibraryFolder'
training_control_samples = {
    'CNTRL_182_s4': 'SLX_19285',
    'CNTRL_193_s2': 'SLX_19285',
    'CNTRL_181_s7': 'SLX_19285',
    'CNTRL_175_s3': 'SLX_20125',
    'CNTRL_190_s4': 'SLX_20125',
    'CNTRL_164_s6': 'SLX_20125',
    'CNTRL_189_s4': 'SLX_20127',
    'CNTRL_162_s5': 'SLX_20127',
    'CNTRL_168_s4': 'SLX_20127'
}

In [ ]:
output_dir = "Data_files/mCA_calling/Simulated_data/Simulated_samples_TRAINING_set"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# RUN THE SIMULATION...
# ==========================================
print("🚀 Starting Batch Simulation (v4.1 - Seed Recorded)...")
manifest_data = [] # Collect data during loop now
used_chroms = set()

for i, (sample, lib_folder) in enumerate(training_control_samples.items()):
    print(f"\n📂 Processing Sample: {sample} (in {lib_folder})")
    
    # Locate Files
    snp_name_1 = f"{sample}_watson_code_SSCS_variant_calling_only_SNPs_annovar_annotated.txt"
    snp_path = os.path.join(base_path, lib_folder, sample, snp_name_1)
    if not os.path.exists(snp_path):
        snp_name_2 = f"{sample}_CNV_watson_code_SSCS_variant_calling_only_SNPs_annovar_annotated.txt"
        snp_path = os.path.join(base_path, lib_folder, sample, snp_name_2)
        if not os.path.exists(snp_path):
             print(f"   ⚠️ SNP file missing: {snp_path}")
             continue
    
    lrr_folder = os.path.join(base_path, lib_folder, sample, "PON_normalised_log2ratios_Feb2026")
    lrr_path = os.path.join(lrr_folder, f"{sample}_PON_normalised_read_depths_and_LRR.txt")
    if not os.path.exists(lrr_path):
        lrr_path = os.path.join(base_path, lib_folder, sample, f"{sample}_PON_normalised_read_depths_and_LRR.txt")
        if not os.path.exists(lrr_path):
            print(f"   ⚠️ LRR file missing: {lrr_path}")
            continue
        
    try:
        df_snps_raw = pd.read_csv(snp_path, sep='\t')
        df_lrr_raw = pd.read_csv(lrr_path, sep='\t', skiprows=5)
        df_lrr_raw.columns = df_lrr_raw.columns.str.strip()
        df_lrr_raw['chromosome'] = df_lrr_raw['chromosome'].astype(str).apply(lambda x: x if x.startswith('chr') else f'chr{x}')
        df_lrr_raw.set_index(['chromosome', 'start', 'stop', 'band'], inplace=True)
    except Exception as e:
        print(f"   ❌ Error reading files: {e}")
        continue

    # Simulation Loop
    sample_seed = MASTER_SEED + (i * 1000) 
    # Pick clean chromosomes avoiding pre-existing mCAs
    selected_chroms = get_clean_chromosomes(sample, control_events, n_chroms=3, seed=sample_seed, used_chroms=used_chroms)
    used_chroms.update(selected_chroms)
    
    print(f"   🎲 Selected Chromosomes: {selected_chroms}")
    affected = control_events.get(sample, [])
    if affected:
        print(f"   ⚠️ Avoided chromosomes with pre-existing events: {affected}")
    
    sim_count = 0
    
    for chrom in selected_chroms:
        for geom in simulation_geometries:
            coord_seed = sample_seed + sim_count + 100
            start, end = get_coordinates(chrom, geom['len'], geom['type'], coord_seed, excluded_arms, bed_df)
            
            if start == 0 and end == 0: continue
            
            len_label = "WholeArm" if geom['len'] == 0 else f"{geom['len']}Mb"
            
            for mca in mca_types:
                # Generate haplotype assignment ONCE per (chrom, geom, mca_type).
                # This ensures the same physical haplotype is amplified across all
                # CF levels, matching real biology where the clone's haplotype
                # structure is fixed regardless of clone size.
                snp_chrom = chrom if chrom.startswith('chr') else f"chr{chrom}"
                family_het_mask = (df_snps_raw['chromosome'] == snp_chrom) & \
                                  (df_snps_raw['position'] >= start) & \
                                  (df_snps_raw['position'] <= end) & \
                                  (df_snps_raw['VAF'] > 0.2) & (df_snps_raw['VAF'] < 0.8)
                n_hets = family_het_mask.sum()
                family_haplotypes = np.random.RandomState(coord_seed).binomial(1, 0.5, n_hets) if n_hets > 0 else None
                
                for frac in fractions:
                    final_seed = coord_seed + sim_count
                    sim_id = f"{mca}_{frac*100:g}pct_{chrom}_{len_label}_{geom['type']}"
                    
                    # Run Simulation — same haplotype assignment for all CFs
                    run_simulation(sample, chrom, start, end, mca, frac, 
                                   df_snps_raw, df_lrr_raw, sim_id, final_seed,
                                   haplotype_assignment=family_haplotypes)
                    
                    # Add to Manifest Memory
                    manifest_data.append({
                        'Sample': sample,
                        'Type': mca,
                        'Fraction_Percent': frac,
                        'Chromosome': chrom,
                        'Start_bp': start,
                        'End_bp': end,
                        'Length_Category': len_label,
                        'Geometry': geom['type'],
                        'Seed_Used': final_seed,
                        'File_Name': f"{sample}_{sim_id}_PON_normalised_read_depths_and_LRR.txt"
                    })
                    
                    sim_count += 1
    
    print(f"   ✨ Generated {sim_count} simulations for {sample}")

# ==========================================
# 5. SAVE MANIFEST
# ==========================================
print("\n📝 Saving Truth Manifest...")
if manifest_data:
    df_manifest = pd.DataFrame(manifest_data)
    df_manifest = df_manifest.sort_values(['Sample', 'Chromosome', 'Type'])
    df_manifest.to_csv(os.path.join(output_dir, "simulation_manifest.csv"), index=False)
    print(f"✅ Manifest saved to: {output_dir}/simulation_manifest.csv")
else:
    print("⚠️ No simulations ran.")

print("\n✅ All Batch Simulations Complete.")

## TEST set of samples

In [ ]:
# ==========================================
# 1. LOAD DATA
# ==========================================
print("⏳ Loading PON Master Matrix (for Noise/CV calculations)...")
if not os.path.exists(pon_matrix_path):
    print(f"❌ Error: PON Matrix not found at {pon_matrix_path}")
    raise SystemExit

# Load Matrix (used ONLY for CV)
pon_df = pd.read_csv(pon_matrix_path, sep='\t', index_col=[0, 1, 2, 3])

# Calculate CV (Noise) from the Matrix
# CV = StdDev / Mean (The mean here is ~1.0, but we calculate it to be precise)
sample_cvs = pon_df.std(axis=0) / (pon_df.mean(axis=0) + 1e-6)

print(f"✅ Loaded Matrix. Calculating Raw Depths for {len(sample_cvs)} samples...")

# ==========================================
# 2. RETRIEVE RAW DEPTHS FROM SOURCE FILES
# ==========================================
sample_metadata = {}

for lib in libraries:
    lib_path = os.path.join(base_path, lib)
    if not os.path.exists(lib_path): continue
    
    # Get all sample folders
    samples = [d for d in os.listdir(lib_path) if os.path.isdir(os.path.join(lib_path, d))]
    
    for s in samples:
        # Skip excluded samples
        if s in excluded_samples:
            continue

        # Only process if this sample is in our PON
        if s not in sample_cvs.index: continue
            
        # Find the raw data file
        cnv_folder = os.path.join(lib_path, s, "CNV_read_depths")
        if not os.path.exists(cnv_folder): continue
        
        # Look for the file
        files = [f for f in os.listdir(cnv_folder) if s in f and f.endswith("sample_normalised_read_depths.txt") and "PON" not in f]
        if not files: continue
        
        # Update the file path for THIS sample
        file_path = os.path.join(cnv_folder, files[0])
        
        try:
            df = pd.read_csv(file_path, sep='\t', skiprows=5)
            df.columns = df.columns.str.strip()
                
            if 'mean_region_depth' in df.columns:
                raw_depth = df['mean_region_depth'].mean()
                
                sample_metadata[s] = {
                    'Library': lib,
                    'Raw_Depth': raw_depth,
                    'CV': sample_cvs[s]
                }
        except:
            pass

# Convert to DataFrame
stats_df = pd.DataFrame.from_dict(sample_metadata, orient='index')

# ==========================================
# 3. SELECT TEST SAMPLES (CLOSEST TO MEDIAN, EXCLUDING TRAINING SET)
# ==========================================
test_selected_samples = []
training_sample_names = list(training_control_samples.keys())

print("\n🎯 Selecting TEST Samples (Excluding Training Controls)...")

for lib in libraries:
    lib_df = stats_df[stats_df['Library'] == lib].copy()
    if lib_df.empty: continue
    
    # --- EXCLUDE training controls from candidates ---
    lib_df = lib_df[~lib_df.index.isin(training_sample_names)]
    if lib_df.empty: continue
        
    # Calculate Centroids (Medians) — on the remaining pool
    median_depth = lib_df['Raw_Depth'].median()
    median_cv = lib_df['CV'].median()
    
    # Normalize for distance calculation
    d_norm = (lib_df['Raw_Depth'] - lib_df['Raw_Depth'].mean()) / lib_df['Raw_Depth'].std()
    c_norm = (lib_df['CV'] - lib_df['CV'].mean()) / lib_df['CV'].std()
    md_norm = (median_depth - lib_df['Raw_Depth'].mean()) / lib_df['Raw_Depth'].std()
    mc_norm = (median_cv - lib_df['CV'].mean()) / lib_df['CV'].std()
    
    # Distance
    lib_df['dist'] = np.sqrt((d_norm - md_norm)**2 + (c_norm - mc_norm)**2)
    
    # Pick top 3
    top_3 = lib_df.sort_values('dist').head(3).index.tolist()
    test_selected_samples.extend(top_3)
    
    print(f"   📂 {lib}: Median Depth={median_depth:.0f}, Median CV={median_cv:.4f}")
    print(f"      -> Test Selected: {top_3}")

# ==========================================
# 4. PLOTTING (TRAINING + TEST)
# ==========================================
fig = plt.figure(figsize=(12, 12))
gs = gridspec.GridSpec(2, 2, height_ratios=[3, 1], hspace=0.3)

# --- MAIN SCATTER ---
ax_main = plt.subplot(gs[0, :])
sns.scatterplot(
    data=stats_df, x='Raw_Depth', y='CV', hue='Library', style='Library',
    s=100, alpha=0.6, edgecolor='k', ax=ax_main
)

# Training samples — black circles
training_data = stats_df.loc[stats_df.index.isin(training_sample_names)]
ax_main.scatter(
    training_data['Raw_Depth'], training_data['CV'], 
    s=400, facecolors='none', edgecolors='black', linewidth=2, zorder=10,
    label='Training Set'
)
for s in training_sample_names:
    if s in stats_df.index:
        row = stats_df.loc[s]
        ax_main.text(
            row['Raw_Depth'], row['CV'], f"  {s}", 
            fontsize=9, fontweight='bold', ha='left', va='center', color='black'
        )

# Test samples — red circles
test_data = stats_df.loc[test_selected_samples]
ax_main.scatter(
    test_data['Raw_Depth'], test_data['CV'], 
    s=400, facecolors='none', edgecolors=grey3, linewidth=2, zorder=10,
    label='Test Set'
)
for s in test_selected_samples:
    row = stats_df.loc[s]
    ax_main.text(
        row['Raw_Depth'], row['CV'], f"  {s}", 
        fontsize=8, fontweight='bold', ha='left', va='center', color=grey4
    )

# ax_main.set_title("Training & Test Set Selection: mean depth vs noise", fontsize=8, fontweight='bold')
ax_main.set_xlabel("mean sequencing depth", fontsize=8)
ax_main.set_ylabel("CV (noise from PON)", fontsize=8)
ax_main.legend(fontsize=8)
ax_main.grid(True, linestyle='--', alpha=0.4)

# --- HISTOGRAMS ---
all_marked = training_sample_names + test_selected_samples

ax_h1 = plt.subplot(gs[1, 0])
sns.kdeplot(data=stats_df, x='Raw_Depth', hue='Library', fill=True, ax=ax_h1, alpha=0.3)
for s in training_sample_names:
    if s in stats_df.index:
        ax_h1.axvline(stats_df.loc[s, 'Raw_Depth'], color='black', linestyle='--', alpha=0.5)
for s in test_selected_samples:
    ax_h1.axvline(stats_df.loc[s, 'Raw_Depth'], color=grey3, linestyle='--', alpha=0.5)
ax_h1.set_title("Distribution of mean sequencing depths")
ax_h1.set_xlabel("mean sequencing depth", fontsize=8)
ax_h1.set_ylabel("density", fontsize=8)

ax_h2 = plt.subplot(gs[1, 1])
sns.kdeplot(data=stats_df, x='CV', hue='Library', fill=True, ax=ax_h2, alpha=0.3)
for s in training_sample_names:
    if s in stats_df.index:
        ax_h2.axvline(stats_df.loc[s, 'CV'], color='black', linestyle='--', alpha=0.5)
for s in test_selected_samples:
    ax_h2.axvline(stats_df.loc[s, 'CV'], color=grey4, linestyle='--', alpha=0.5)
ax_h2.set_title("Distribution of noise (CV)")
ax_h2.set_xlabel("CV (noise from PON)", fontsize=8)
ax_h2.set_ylabel("density", fontsize=8)

for ax in [ax_main, ax_h1, ax_h2]:
    ax.tick_params(axis='both', labelsize=8)
    for axis in ['left', 'bottom']:
        ax.spines[axis].set_linewidth(1.5)
        ax.spines[axis].set_color(grey3)
    for axis in ['top', 'right']:
        ax.spines[axis].set_visible(False)
    ax.yaxis.set_tick_params(width=1.5, color = grey3, length = 6, which = 'major')
    ax.xaxis.set_tick_params(width=0, color = grey3, length = 6, which = 'major')

plt.savefig("Training_and_Test_Control_Selection.pdf", bbox_inches='tight')
plt.show()

# Export test set for next step
with open("test_selected_controls.txt", "w") as f:
    f.write("\n".join(test_selected_samples))

In [ ]:
# --- CONTROL SAMPLES TO USE ---
# Format: 'SampleName': 'LibraryFolder'
test_control_samples = {
    'CNTRL_200_s2': 'SLX_19285',
    'CNTRL_204_s5': 'SLX_19285',
    'CNTRL_186_s4': 'SLX_19285',
    'CNTRL_171_s2': 'SLX_20125',
    'CNTRL_194_s8': 'SLX_20125',
    'CNTRL_173_s3': 'SLX_20125',
    'CNTRL_004_s10': 'SLX_20127',
    'CNTRL_198_s2': 'SLX_20127',
    'CNTRL_005_s9': 'SLX_20127'
}

In [ ]:
output_dir = "Data_files/mCA_calling/Simulated_data/Simulated_samples_TEST_set"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# RUN THE SIMULATION...
# ==========================================
print("🚀 Starting Batch Simulation (v4.1 - Seed Recorded)...")
manifest_data = [] # Collect data during loop now
used_chroms = set()

for i, (sample, lib_folder) in enumerate(test_control_samples.items()):
    print(f"\n📂 Processing Sample: {sample} (in {lib_folder})")
    
    # Locate Files
    snp_name_1 = f"{sample}_watson_code_SSCS_variant_calling_only_SNPs_annovar_annotated.txt"
    snp_path = os.path.join(base_path, lib_folder, sample, snp_name_1)
    if not os.path.exists(snp_path):
        snp_name_2 = f"{sample}_CNV_watson_code_SSCS_variant_calling_only_SNPs_annovar_annotated.txt"
        snp_path = os.path.join(base_path, lib_folder, sample, snp_name_2)
        if not os.path.exists(snp_path):
             print(f"   ⚠️ SNP file missing: {snp_path}")
             continue
    
    lrr_folder = os.path.join(base_path, lib_folder, sample, "PON_normalised_log2ratios_Feb2026")
    lrr_path = os.path.join(lrr_folder, f"{sample}_PON_normalised_read_depths_and_LRR.txt")
    if not os.path.exists(lrr_path):
        lrr_path = os.path.join(base_path, lib_folder, sample, f"{sample}_PON_normalised_read_depths_and_LRR.txt")
        if not os.path.exists(lrr_path):
            print(f"   ⚠️ LRR file missing: {lrr_path}")
            continue
        
    try:
        df_snps_raw = pd.read_csv(snp_path, sep='\t')
        df_lrr_raw = pd.read_csv(lrr_path, sep='\t', skiprows=5)
        df_lrr_raw.columns = df_lrr_raw.columns.str.strip()
        df_lrr_raw['chromosome'] = df_lrr_raw['chromosome'].astype(str).apply(lambda x: x if x.startswith('chr') else f'chr{x}')
        df_lrr_raw.set_index(['chromosome', 'start', 'stop', 'band'], inplace=True)
    except Exception as e:
        print(f"   ❌ Error reading files: {e}")
        continue

    # Simulation Loop
    sample_seed = MASTER_SEED + (i * 1000) 
    # Pick clean chromosomes avoiding pre-existing mCAs
    selected_chroms = get_clean_chromosomes(sample, control_events, n_chroms=3, seed=sample_seed, used_chroms=used_chroms)
    used_chroms.update(selected_chroms)
    
    print(f"   🎲 Selected Chromosomes: {selected_chroms}")
    affected = control_events.get(sample, [])
    if affected:
        print(f"   ⚠️ Avoided chromosomes with pre-existing events: {affected}")
    
    sim_count = 0
    
    for chrom in selected_chroms:
        for geom in simulation_geometries:
            coord_seed = sample_seed + sim_count + 100
            start, end = get_coordinates(chrom, geom['len'], geom['type'], coord_seed, excluded_arms, bed_df)
            
            if start == 0 and end == 0: continue
            
            len_label = "WholeArm" if geom['len'] == 0 else f"{geom['len']}Mb"
            
            for mca in mca_types:
                # Generate haplotype assignment ONCE per (chrom, geom, mca_type).
                # This ensures the same physical haplotype is amplified across all
                # CF levels, matching real biology where the clone's haplotype
                # structure is fixed regardless of clone size.
                snp_chrom = chrom if chrom.startswith('chr') else f"chr{chrom}"
                family_het_mask = (df_snps_raw['chromosome'] == snp_chrom) & \
                                  (df_snps_raw['position'] >= start) & \
                                  (df_snps_raw['position'] <= end) & \
                                  (df_snps_raw['VAF'] > 0.2) & (df_snps_raw['VAF'] < 0.8)
                n_hets = family_het_mask.sum()
                family_haplotypes = np.random.RandomState(coord_seed).binomial(1, 0.5, n_hets) if n_hets > 0 else None
                
                for frac in fractions:
                    final_seed = coord_seed + sim_count
                    sim_id = f"{mca}_{frac*100:g}pct_{chrom}_{len_label}_{geom['type']}"
                    
                    # Run Simulation — same haplotype assignment for all CFs
                    run_simulation(sample, chrom, start, end, mca, frac, 
                                   df_snps_raw, df_lrr_raw, sim_id, final_seed,
                                   haplotype_assignment=family_haplotypes)
                    
                    # Add to Manifest Memory
                    manifest_data.append({
                        'Sample': sample,
                        'Type': mca,
                        'Fraction_Percent': frac,
                        'Chromosome': chrom,
                        'Start_bp': start,
                        'End_bp': end,
                        'Length_Category': len_label,
                        'Geometry': geom['type'],
                        'Seed_Used': final_seed,
                        'File_Name': f"{sample}_{sim_id}_PON_normalised_read_depths_and_LRR.txt"
                    })
                    
                    sim_count += 1
    
    print(f"   ✨ Generated {sim_count} simulations for {sample}")

# ==========================================
# 5. SAVE MANIFEST
# ==========================================
print("\n📝 Saving Truth Manifest...")
if manifest_data:
    df_manifest = pd.DataFrame(manifest_data)
    df_manifest = df_manifest.sort_values(['Sample', 'Chromosome', 'Type'])
    df_manifest.to_csv(os.path.join(output_dir, "simulation_manifest.csv"), index=False)
    print(f"✅ Manifest saved to: {output_dir}/simulation_manifest.csv")
else:
    print("⚠️ No simulations ran.")

print("\n✅ All Batch Simulations Complete.")